# Step 8 (Kaggle CPU) — DFT training set for fine-tuning (BaTiO3 / PbTiO3)
The MLIPs are trained on PBE (most models) or r2SCAN. To separate the error of the machine-learning model from the error of the DFT functional, we compute the same quantities with DFT ourselves.

**Run on Kaggle:** Settings (right panel) → Internet **On** (needs phone verification), Accelerator **None (CPU)**. Then click **Save Version → Save & Run All (Commit)**. The run continues on Kaggle's servers even if you close the browser (max 12 h). When it is finished, open the version → **Output** tab → download `train_<MATERIAL>.extxyz`. Run twice: MATERIAL = "BaTiO3" and "PbTiO3" (cell 3), in two notebooks.


In [ ]:
# 1. Install Quantum ESPRESSO from conda-forge (3-5 min)
import os, subprocess
os.makedirs('/tmp/k', exist_ok=True)
if not os.path.exists('/tmp/k/qe/bin/pw.x'):
    !wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp/k bin/micromamba
    !/tmp/k/bin/micromamba create -y -q -p /tmp/k/qe -c conda-forge qe > /tmp/k/mamba.log 2>&1; tail -3 /tmp/k/mamba.log
PW = '/tmp/k/qe/bin/pw.x'
assert os.path.exists(PW), 'Quantum ESPRESSO not installed - send the output above to Claude'
# make sure all shared libraries are found (fix soname mismatches, e.g. libelpa)
import glob, re
os.environ['LD_LIBRARY_PATH'] = '/tmp/k/qe/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'; os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'
def missing_libs():
    out = subprocess.run(f'ldd {PW}', shell=True, capture_output=True, text=True).stdout
    return re.findall(r'(\S+) => not found', out)
for lib in missing_libs():
    stem = lib.split('.so')[0]
    cand = sorted(glob.glob(f'/tmp/k/qe/lib/{stem}*.so*'))
    print('missing', lib, '-> candidates', [os.path.basename(c) for c in cand])
    if cand: os.symlink(cand[-1], f'/tmp/k/qe/lib/{lib}')
if missing_libs():
    print('installing missing libraries:', missing_libs())
    subprocess.run('/tmp/k/bin/micromamba install -y -q -p /tmp/k/qe -c conda-forge elpa scalapack > /tmp/k/mamba2.log 2>&1', shell=True)
    for lib in missing_libs():
        cand = sorted(glob.glob(f"/tmp/k/qe/lib/{lib.split('.so')[0]}*.so*"))
        if cand: os.symlink(cand[-1], f'/tmp/k/qe/lib/{lib}')
print('still missing:', missing_libs())
assert not missing_libs(), 'Some libraries are still missing - send this output to Claude'
!pip install -q "ase>=3.23" spglib
MPI = None  # serial + OpenMP: MPI was about 4x slower on Kaggle
print('pw.x =', PW, '| mpirun =', MPI, '| CPUs =', os.cpu_count())


In [ ]:
# 2. Pseudopotentials (GBRV, fallback PseudoDojo)
import os, glob, subprocess, re
OUT = '/kaggle/working'  # only results here; os.makedirs(OUT, exist_ok=True)
SOURCES = {
 'gbrv': ('https://www.physics.rutgers.edu/gbrv/all_{xc}_UPF_v1.5.tar.gz', dict(ecutwfc=45, ecutrho=360)),
 'dojo': ('http://www.pseudo-dojo.org/pseudos/nc-sr-04_{xc}_standard_upf.tgz', dict(ecutwfc=85, ecutrho=340)),
}
PSEUDO = {}
def find_upf(d, el):
    pat = re.compile(rf'^{el}([_.-]|$)', re.I)
    return sorted(f for f in glob.glob(d + '/**/*', recursive=True)
                  if f.lower().endswith('.upf') and pat.match(os.path.basename(f)))
for xc in ['pbe']:
    for src, (url, cut) in SOURCES.items():
        d = f'/tmp/k/pseudo_{xc}_{src}'
        if not os.path.isdir(d) or not glob.glob(d + '/**/*.*', recursive=True):
            os.makedirs(d, exist_ok=True)
            r = subprocess.run(f'wget -q --tries=3 --timeout=60 "{url.format(xc=xc)}" -O /tmp/k/{xc}_{src}.tgz && tar xzf /tmp/k/{xc}_{src}.tgz -C {d}', shell=True)
        ok = all(find_upf(d, el) for el in ['Ba','Pb','Ti','O'])
        print(f'{xc:6s} {src}: download {"OK" if ok else "FAILED"}')
        if ok: PSEUDO[xc] = (d, cut, src); break
print('using:', {k: v[2] for k, v in PSEUDO.items()})
assert 'pbe' in PSEUDO, 'No PBE pseudopotentials could be downloaded - send this output to Claude'


In [ ]:
# 3. Configurations and single-point DFT (energy, forces, stress)
import numpy as np, time, glob
from ase import Atoms
from ase.io import write, read
MATERIAL = "PbTiO3"     # BaTiO3 already done
A = {"BaTiO3": "Ba", "PbTiO3": "Pb"}[MATERIAL]
A0 = {"BaTiO3": 4.027, "PbTiO3": 3.966}[MATERIAL]       # our DFT-PBE cubic lattice constants
NP = os.cpu_count(); rng = np.random.default_rng(2026)
xc = "pbe"
def pseudo(el):
    return find_upf(PSEUDO[xc][0], el)[-1]

def unit(a=A0, d=(0, 0, 1), amp=0.0, eps=(0, 0, 0)):
    frac = [[0,0,0],[.5,.5,.5],[.5,.5,0],[.5,0,.5],[0,.5,.5]]
    at = Atoms([A, "Ti", "O", "O", "O"], scaled_positions=frac, cell=np.eye(3)*a, pbc=True)
    d = np.array(d, float); d /= np.linalg.norm(d); p = at.get_positions()
    p[1] += amp*d; p[2:] -= 0.6*amp*d; at.set_positions(p)          # Slater-like polar mode
    at.set_cell(at.cell[:] @ (np.eye(3) + np.diag(eps)), scale_atoms=True)
    return at

CONF = []
for d in [(0, 0, 1), (1, 1, 0), (1, 1, 1)]:                       # double-well profiles
    for amp in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
        if amp == 0.0 and d != (0, 0, 1): continue
        CONF.append((f"well_{''.join(map(str, d))}_{amp:.2f}", unit(d=d, amp=amp)))
for amp in [0.1, 0.2, 0.3]:                                          # polar + tetragonal strain
    for e in [0.02, 0.05]:
        CONF.append((f"strain_{amp:.1f}_{e:.2f}", unit(d=(0, 0, 1), amp=amp, eps=(-e/3, -e/3, e))))
for k in range(8):                                                   # rattled 2x2x2 supercells
    base = unit(d=[(0, 0, 1), (1, 1, 1)][k % 2], amp=[0.0, 0.1, 0.2][k % 3],
                eps=tuple(rng.normal(0, 0.01, 3))).repeat((2, 2, 2))
    base.positions += rng.normal(0, [0.04, 0.07, 0.10][k % 3], base.positions.shape)
    CONF.append((f"rattle_{k:02d}", base))
print(len(CONF), "configurations")

def kgrid(at, density=24.0):
    if len(at) > 10: density = 16.0          # 2x2x2 k-points for the 40-atom supercells
    return tuple(max(1, int(np.ceil(density / np.linalg.norm(v)))) for v in at.cell)

def scf(at, tag):
    wd = f"/tmp/k/train/{tag}"; os.makedirs(wd, exist_ok=True)
    pp = {el: os.path.basename(pseudo(el)) for el in set(at.get_chemical_symbols())}
    inp = dict(control=dict(calculation="scf", prefix="x", pseudo_dir=os.path.dirname(pseudo("O")), outdir=wd,
                            tprnfor=True, tstress=True),
               system=dict(occupations="fixed", **PSEUDO[xc][1]), electrons=dict(conv_thr=1e-9, mixing_beta=0.4))
    write(f"{wd}/pw.in", at, format="espresso-in", input_data=inp, pseudopotentials=pp, kpts=kgrid(at))
    subprocess.run(f"cd {wd} && OMP_NUM_THREADS={NP} {PW} -in pw.in > pw.out 2>&1", shell=True)
    return read(f"{wd}/pw.out", format="espresso-out", index=-1)



In [ ]:
# 4. Isolated atoms (reference energies E0 for fine-tuning) - computed first
E0F = "/kaggle/working/E0_atoms.json"
import json
E0 = json.load(open(E0F)) if os.path.exists(E0F) else {}
for el in ["O", "Ti", "Ba", "Pb"]:
    if el in E0: continue
    wd = f"/tmp/k/atom/{el}"; os.makedirs(wd, exist_ok=True)
    at = Atoms(el, positions=[[5.0, 5.2, 5.4]], cell=[10.0, 10.5, 11.0], pbc=True)
    inp = dict(control=dict(calculation="scf", prefix="x", pseudo_dir=os.path.dirname(pseudo("O")), outdir=wd),
               system=dict(occupations="smearing", smearing="gaussian", degauss=0.005, nspin=2,
                           starting_magnetization={el: 0.5}, **PSEUDO[xc][1]),
               electrons=dict(conv_thr=1e-8, mixing_beta=0.2, electron_maxstep=300))
    write(f"{wd}/pw.in", at, format="espresso-in", input_data=inp, pseudopotentials={el: os.path.basename(pseudo(el))}, kpts=None)
    subprocess.run(f"cd {wd} && OMP_NUM_THREADS={NP} {PW} -in pw.in > pw.out 2>&1", shell=True)
    try:
        E0[el] = read(f"{wd}/pw.out", format="espresso-out").get_potential_energy(); print(el, E0[el])
    except Exception as e:
        print("isolated atom failed", el, repr(e)[:200])
    json.dump(E0, open(E0F, "w"), indent=1)
print(E0)


In [ ]:
# 5. Training configurations (stops cleanly after 11 h)
T_START = time.time()
OUTF = f"/kaggle/working/train_{MATERIAL}.extxyz"
done_tags = {a.info["tag"] for a in read(OUTF, index=":")} if os.path.exists(OUTF) else set()
for tag, at in CONF:
    if tag in done_tags: continue
    if time.time() - T_START > 11 * 3600: print("time budget reached"); break
    t0 = time.time()
    try:
        res = scf(at, tag)
    except Exception as e:
        print("FAILED", tag, repr(e)[:200]); continue
    out = at.copy(); out.info.update(tag=tag, material=MATERIAL, REF_energy=res.get_potential_energy(),
                                     REF_stress=res.get_stress(voigt=False).reshape(-1))
    out.arrays["REF_forces"] = res.get_forces()
    write(OUTF, out, append=True, format="extxyz")
    print(f"{tag:22s} natoms={len(at):3d} E={res.get_potential_energy():.4f} eV  maxF={np.abs(res.get_forces()).max():.3f}  {time.time()-t0:.0f}s", flush=True)
